In [12]:
import pandas as pd
import numpy as np

In [8]:
df = pd.read_csv(r"C:\Users\TGX-100\Documents\GitHub\Urban-Traffic-Flow-Prediction\dataset\processed\GA0151_intersection.csv")

In [45]:
df['date'] = pd.to_datetime(df['date'])

In [46]:
df = df.sort_values(by=["date", "hour"]).reset_index(drop=True)

In [47]:
df.head()

,date,hour,GA0151_A,GA0151_C,GA0151_D,day_of_week,day_of_month,month
0,2019-10-01,0,6,16,15,1,1,10
1,2019-10-01,1,4,16,8,1,1,10
2,2019-10-01,2,4,12,14,1,1,10
3,2019-10-01,3,0,16,10,1,1,10
4,2019-10-01,4,4,21,15,1,1,10


## Time based features

In [51]:
df['day_of_week'] = df['date'].dt.dayofweek
df['day_of_month'] = df['date'].dt.day
df['month'] = df['date'].dt.month
df['year'] = df['date'].dt.year
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

In [52]:
df

,date,hour,GA0151_A,GA0151_C,GA0151_D,day_of_week,day_of_month,month,is_weekend,year
0,2019-10-01,0,6,16,15,1,1,10,0,2019
1,2019-10-01,1,4,16,8,1,1,10,0,2019
2,2019-10-01,2,4,12,14,1,1,10,0,2019
3,2019-10-01,3,0,16,10,1,1,10,0,2019
4,2019-10-01,4,4,21,15,1,1,10,0,2019
...,...,...,...,...,...,...,...,...,...,...
33639,2023-09-30,19,37,205,146,5,30,9,1,2023
33640,2023-09-30,20,33,159,154,5,30,9,1,2023
33641,2023-09-30,21,43,152,145,5,30,9,1,2023
33642,2023-09-30,22,36,159,119,5,30,9,1,2023


## Cyclical Time features

In [53]:
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

df['day_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['day_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

In [54]:
df

,date,hour,GA0151_A,GA0151_C,GA0151_D,day_of_week,day_of_month,month,is_weekend,year,hour_sin,hour_cos,day_sin,day_cos,month_sin,month_cos
0,2019-10-01,0,6,16,15,1,1,10,0,2019,0.000000,1.000000,0.781831,0.623490,-0.866025,5.000000e-01
1,2019-10-01,1,4,16,8,1,1,10,0,2019,0.258819,0.965926,0.781831,0.623490,-0.866025,5.000000e-01
2,2019-10-01,2,4,12,14,1,1,10,0,2019,0.500000,0.866025,0.781831,0.623490,-0.866025,5.000000e-01
3,2019-10-01,3,0,16,10,1,1,10,0,2019,0.707107,0.707107,0.781831,0.623490,-0.866025,5.000000e-01
4,2019-10-01,4,4,21,15,1,1,10,0,2019,0.866025,0.500000,0.781831,0.623490,-0.866025,5.000000e-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33639,2023-09-30,19,37,205,146,5,30,9,1,2023,-0.965926,0.258819,-0.974928,-0.222521,-1.000000,-1.836970e-16
33640,2023-09-30,20,33,159,154,5,30,9,1,2023,-0.866025,0.500000,-0.974928,-0.222521,-1.000000,-1.836970e-16
33641,2023-09-30,21,43,152,145,5,30,9,1,2023,-0.707107,0.707107,-0.974928,-0.222521,-1.000000,-1.836970e-16
33642,2023-09-30,22,36,159,119,5,30,9,1,2023,-0.500000,0.866025,-0.974928,-0.222521,-1.000000,-1.836970e-16


### Lag Features for sensor

In [55]:
sensors = ["GA0151_A", "GA0151_C", "GA0151_D"]

for sensor in sensors:
    # Lag Features
    df[f"{sensor}_lag_1"] = df[sensor].shift(1)
    df[f"{sensor}_lag_2"] = df[sensor].shift(2)
    df[f"{sensor}_lag_3"] = df[sensor].shift(3)
    df[f"{sensor}_lag_6"] = df[sensor].shift(6)
    df[f"{sensor}_lag_12"] = df[sensor].shift(12)
    df[f"{sensor}_lag_24"] = df[sensor].shift(24)
    df[f"{sensor}_lag_48"] = df[sensor].shift(48)
    df[f"{sensor}_lag_168"] = df[sensor].shift(168)

    # Rolling Mean
    df[f"{sensor}_rolling_mean_3"] = df[sensor].shift(1).rolling(3).mean()
    df[f"{sensor}_rolling_mean_6"] = df[sensor].shift(1).rolling(6).mean()
    df[f"{sensor}_rolling_mean_12"] = df[sensor].shift(1).rolling(12).mean()
    df[f"{sensor}_rolling_mean_24"] = df[sensor].shift(1).rolling(24).mean()

    # Rolling Standard Deviation
    df[f"{sensor}_rolling_std_6"] = df[sensor].shift(1).rolling(6).std()
    df[f"{sensor}_rolling_std_24"] = df[sensor].shift(1).rolling(24).std()

    # Difference Features
    df[f"{sensor}_diff_1"] = df[sensor] - df[f"{sensor}_lag_1"]
    df[f"{sensor}_diff_24"] = df[sensor] - df[f"{sensor}_lag_24"]

